In [1]:
import torch

In [2]:
torch.manual_seed(42)

Architecture:
2 hidden layers
relu activation function
tanh activation function
sigmoid output

In [ ]:
from torch import float32


class NeuralNetwork():
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, bias_value):
        self.W1 = torch.randn(input_size, hidden_size1)
        self.W2 = torch.randn(hidden_size1, hidden_size2)
        self.W3 = torch.randn(hidden_size2, output_size)
        self.b1 = torch.full((hidden_size1,), bias_value)
        self.b2 = torch.full((hidden_size2,), bias_value)
        self.b3 = torch.full((output_size,), bias_value)
    
    def relu(self, r):
        return torch.relu(r)
    
    def tanh(self, t):
        return torch.tanh(t)

    def tanh_grad(self, t):
        return 1 - t ** 2

    def sigmoid(self, s):
        return torch.sigmoid(s)
    
    def accuracy(self, pred, y):
        labels = torch.round(pred).float()
        return torch.mean((labels == y).float())


    def train(self, X, y, iterations = 10):
        losses = []
        
        for i in range(iterations):
            pred, cache = self.forward(X)
            iter_loss = self.loss(pred, y)
            self.backward(X, pred, y, cache)
            losses.append(iter_loss)

            print(f"Loss: {iter_loss}, accuracy: {self.accuracy(pred, y)}")
        
        return losses

    
    def forward(self, X):
        A1 = X @ self.W1 + self.b1
        # print(f"A1 shape: {A1.shape}")
        H1 = self.relu(A1)
        # print(f"H1 shape: {H1.shape}")

        A2 = H1 @ self.W2 + self.b2
        # print(f"A2 shape: {A2.shape}")
        H2 = self.tanh(A2)
        # print(f"H2 shape: {H2.shape}")

        A3 = H2 @ self.W3 + self.b3
        # print(f"A3 shape: {A3.shape}")
        p = self.sigmoid(A3)

        cache = (A1, H1, A2, H2, A3)
        # print(p)
        return p, cache
    
    def loss(self, p, y):
        N = y.shape[0]
        return -torch.sum(y * torch.log(p) + (1 - y) * torch.log(1 - p)) / N
    
    def relu_grad(self, r):
        return r * torch.where(r > 0, 1, 0)
    
    def loss_grad(self, p, y):
        N = y.shape[0]
        return -(y / p - (1 - y) / (1 - p)) / N
    
    def sigmoid_grad(self, s):
        return s * (1 - s)

    def adam(self, param, grad, config=None):
        if config is None:
            config = {}

        config.setdefault('learning_rate', 1e-3)
        config.setdefault('beta1', 0.9)
        config.setdefault('beta2', 0.999)
        config.setdefault('epsilon', 1e-8)
        config.setdefault('m', torch.zeros_like(param))
        config.setdefault('v', torch.zeros_like(param))
        config.setdefault('t', 1)

        config['t'] += 1
        
        config['m'] = config['beta1'] * config['m'] + (1 - config['beta1']) * grad
        mt = config['m'] / (1 - config['beta1']**config['t'])

        config['v'] = config['beta2'] * config['v'] + (1 - config['beta2']) * (grad ** 2)
        vt = config['v'] / (1 - config['beta2']**config['t'])

        next_param = param - config['learning_rate'] * mt / (torch.sqrt(vt) + config['epsilon'])
        
        return next_param

    def backward(self, X, pred, y, cache):
        A1, H1, A2, H2, A3 = cache

        dL = self.loss_grad(pred, y) # shape: (input_size, output_size)
        dL_dA3 = self.sigmoid_grad(A3) # shape: (input_size, output_size)

        dW3 = H2.T @ dL_dA3 # shape: (hidden_size2, input_size) @ (input_size, output_size) = (hidden_size2, output_size)
        db3 = torch.sum(dL_dA3, dim = 0) # (output_size, 1)
        dH2 = dL_dA3 @ self.W3.T # (input_size, output_size) @ (output_size, hidden_size_2) = (input_size, hidden_size2)

        dA2 = self.tanh_grad(A2) * dH2

        dW2 = H1.T @ dA2
        db2 = torch.sum(dA2, dim = 0)
        dH1 = dA2 @ self.W2.T

        dA1 = self.relu_grad(A1)
        
        dW1 = X.T @ dA1
        db1 = torch.sum(dA1, dim = 0)

        self.W1 = self.adam(self.W1, dW1)
        self.W2 = self.adam(self.W2, dW2)
        self.W3 = self.adam(self.W3, dW3)

        self.b1 = self.adam(self.b1, db1)
        self.b2 = self.adam(self.b2, db2)
        self.b3 = self.adam(self.b3, db3)
        
        print(self.W1)
        

In [4]:
from torch import float32


X = torch.tensor([
    [1.0, 2.0],
    [2.0, 3.0],
    [3.0, 4.0],
    [4.0, 5.0],
    [5.0, 6.0],
], dtype = float32)

y = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [1.0],
    [0.0]
], dtype = float32)

In [5]:
NN = NeuralNetwork(input_size = X.shape[1], hidden_size1=8, hidden_size2=4, output_size=1, bias_value=-0.1)
predictions, cache = NN.forward(X)
# print(predictions)
print(NN.loss(predictions, y))
print(NN.loss_grad(predictions, y))

tensor(0.6975)
tensor([[-0.4083],
        [ 0.3923],
        [-0.4078],
        [-0.4076],
        [ 0.3931]])


In [8]:
NN = NeuralNetwork(input_size = X.shape[1], hidden_size1=8, hidden_size2=4, output_size=1, bias_value=-0.1)
NN.train(X,y)

Loss: 0.909960925579071, accuracy: 0.4000000059604645
Loss: 0.9089428186416626, accuracy: 0.4000000059604645
Loss: 0.9079219102859497, accuracy: 0.4000000059604645
Loss: 0.9068983197212219, accuracy: 0.4000000059604645
Loss: 0.9058723449707031, accuracy: 0.4000000059604645
Loss: 0.9048439860343933, accuracy: 0.4000000059604645
Loss: 0.9038135409355164, accuracy: 0.4000000059604645
Loss: 0.9027811288833618, accuracy: 0.4000000059604645
Loss: 0.9017465710639954, accuracy: 0.4000000059604645
Loss: 0.9007104635238647, accuracy: 0.4000000059604645


[tensor(0.9100),
 tensor(0.9089),
 tensor(0.9079),
 tensor(0.9069),
 tensor(0.9059),
 tensor(0.9048),
 tensor(0.9038),
 tensor(0.9028),
 tensor(0.9017),
 tensor(0.9007)]